### Residual Network

In [1]:
#importing libraries

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, BatchNormalization, Conv2D, Dense, Flatten, Add
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical

In [7]:
#loading and preprocessing fashion-mnist dataset

(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

train_images = train_images/255.
test_images = test_images/255.

train_images = np.expand_dims(train_images, -1)
test_images = np.expand_dims(test_images, -1)

In [9]:
#creating dataset objects for training and testing sets

train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
train_dataset = train_dataset.batch(32)

test_dataset = tf.data.Dataset.from_tensor_slices((test_images, test_labels))
test_dataset = test_dataset.batch(32)

In [10]:
#dataset label names

image_labels = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

In [15]:
#creating custom layers for residual block

class ResidualBlock(Layer):

    def __init__(self, **kwargs ):
        super(ResidualBlock, self).__init__(**kwargs)

    def build(self, input_shape):
        self.batch_norm_1 = BatchNormalization(input_shape=input_shape)
        self.conv_1 = Conv2D(filters =input_shape[-1], kernel_size=3, padding="same")
        self.batch_norm_2 = BatchNormalization()
        self.conv_2 = Conv2D(filters=input_shape[-1], kernel_size=3, padding="same")

    def call(self, inputs, training=False):
        x = self.batch_norm_1(inputs, training=training)
        x = tf.nn.relu(x)
        x = self.conv_1(x)
        x = self.batch_norm_2(x, training=training)
        x = tf.nn.relu(x)
        
        return Add()([inputs, self.conv_2(x)])
    

In [16]:
#testing cutsom layer

test_model = tf.keras.Sequential([
    ResidualBlock(input_shape=(28,28,1))
        ])
test_model.summary()

C:\Users\mouni\AppData\Local\Temp\ipykernel_2648\1993840464.py:6: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(ResidualBlock, self).__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ residual_block_2                │ (None, 28, 28, 1)      │            28 │
│ (ResidualBlock)                 │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28 (112.00 B)

 Trainable params: 24 (96.00 B)

 Non-trainable params: 4 (16.00 B)

In [17]:
#second custom layer

class FiltersChangResidualBlock(Layer):

    def __init__(self, out_filters, **kwargs):
        super(FiltersChangResidualBlock, self).__init__()
        self.out_filters = out_filters

    def build(self, input_shape):
        self.batch_norm_1 = BatchNormalization(input_shape=input_shape)
        self.conv_1 = Conv2D(filters=input_shape[-1], kernel_size=3, padding="same")
        self.batch_norm_2 = BatchNormalization()
        self.conv_2 = Conv2D(filters=self.out_filters, kernel_size=3, padding="same")
        self.conv_3 = Conv2D(filters=self.out_filters, kernel_size=1)

    def call(self, inputs, training=False):

        x = self.batch_norm_1(inputs, training=training)
        x = tf.nn.relu(x)
        x = self.conv_1(x)
        x = self.batch_norm_2(x, training=training)
        x = tf.nn.relu(x)
        x = self.conv_2(x)
        return Add()([self.conv_3(x), x])




In [18]:
#testing custom layer

test_model = tf.keras.Sequential([
    FiltersChangResidualBlock(out_filters=16, input_shape=(32,32,3))
])
test_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ filters_chang_residual_block    │ ?                      │   0 (unbuilt) │
│ (FiltersChangResidualBlock)     │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [23]:
#custom model creation

class ResNetModel(Model):

    def __init__(self, **kwargs):
        super(ResNetModel, self).__init__(**kwargs)
        self.conv_1 = Conv2D(filters=32, kernel_size=7, strides=2)
        self.res_block = ResidualBlock()
        self.conv_2 = Conv2D(filters=32, kernel_size=3, strides=2)
        self.filter_chg_res_block = FiltersChangResidualBlock(out_filters=64)
        self.flatten = Flatten()
        self.dense = Dense(units=10, activation="softmax")

    def call(self, inputs, training=False):

        x = self.conv_1(inputs)
        x = self.res_block(x, training=training)
        x = self.conv_2(x)
        x = self.filter_chg_res_block(x, out_filters=64)
        x = self.flatten(x)
        return self.dense(x)



In [25]:
#creating the model

resnet_model = ResNetModel()
resnet_model.summary()

Model: "res_net_model_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_10 (Conv2D)              │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_5                │ ?                      │   0 (unbuilt) │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ filters_chang_residual_block_2  │ ?                      │   0 (unbuilt) │
│ (FiltersChangResidualBlock)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [26]:
#defining optimizer and loss function

optimizer_obj = tf.keras.optimizers.Adam(learning_rate=0.001)
loss_obj = tf.keras.losses.SparseCategoricalCrossentropy()

In [27]:
#defining the grad function

@tf.function
def grad(model, inputs, targets, loss):

    with tf.GradientTape() as tape:
        loss = loss(targets, model(inputs))
        grads = tape.gradient(loss, model.trainable_variables)
        return loss, grads